In [1]:
import requests
import json

payload = {'student': 'miren', 'course': 'data_science'}

r_data = requests.post('https://httpbin.org/post', data=payload)
j_data = r_data.json()
print('data form:', j_data.get('form'))
print('data json:', j_data.get('json'))
print('data content-type:', j_data.get('headers', {}).get('Content-Type'))

r_json = requests.post('https://httpbin.org/post', json=payload)
j_json = r_json.json()
print('json form:', j_json.get('form'))
print('json json:', j_json.get('json'))
print('json content-type:', j_json.get('headers', {}).get('Content-Type'))

compare_out = {
    'with_data': {
        'form': j_data.get('form'),
        'json': j_data.get('json'),
        'content_type': j_data.get('headers', {}).get('Content-Type')
    },
    'with_json': {
        'form': j_json.get('form'),
        'json': j_json.get('json'),
        'content_type': j_json.get('headers', {}).get('Content-Type')
    }
}

with open('post_compare.json', 'w', encoding='utf-8') as f:
    json.dump(compare_out, f, indent=2)

data form: {'student': 'miren', 'course': 'data_science'}
data json: None
data content-type: application/x-www-form-urlencoded
json form: {}
json json: {'student': 'miren', 'course': 'data_science'}
json content-type: application/json


In [2]:
users_raw = requests.get('https://jsonplaceholder.typicode.com/users').json()

flat_users = []
for u in users_raw:
    flat_users.append({
        'id': u['id'],
        'name': u['name'],
        'username': u['username'],
        'email': u['email'],
        'city': u['address']['city'],
        'lat': float(u['address']['geo']['lat']),
        'company': u['company']['name']
    })

top3_lat = sorted(flat_users, key=lambda item: item['lat'], reverse=True)[:3]
for item in top3_lat:
    print(item['name'], item['city'], item['lat'])

Patricia Lebsack South Elvis 29.4572
Kurtis Weissnat Howemouth 24.8918
Glenna Reichert Bartholomewsbury 24.6463


In [3]:
import csv

fieldnames = ['id', 'name', 'username', 'email', 'city', 'lat', 'company']
with open('jsonph_users.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(flat_users)

with open('jsonph_users.csv', 'r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    loaded_rows = list(reader)
    cols = reader.fieldnames

print('rows:', len(loaded_rows), 'cols:', len(cols))

rows: 10 cols: 7


In [4]:
posts_raw = requests.get('https://jsonplaceholder.typicode.com/posts').json()

counts_by_user = {}
for p in posts_raw:
    uid = p['userId']
    counts_by_user[uid] = counts_by_user.get(uid, 0) + 1

merged_rows = []
for u in flat_users:
    merged_rows.append({
        'id': u['id'],
        'name': u['name'],
        'city': u['city'],
        'post_count': counts_by_user.get(u['id'], 0)
    })

with open('user_post_counts.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'name', 'city', 'post_count'])
    writer.writeheader()
    writer.writerows(merged_rows)

with open('user_post_counts.json', 'w', encoding='utf-8') as f:
    json.dump(merged_rows, f, indent=2)

total_posts = sum(row['post_count'] for row in merged_rows)
print('total posts:', total_posts)

total posts: 100
